In [ ]:
# 🏆 Desafio — "Meu Analisador Léxico de Mercado"
# Tema 2 — 📦 Rastreio de encomendas (Correios / Mercado Livre)
# Disciplina: Compiladores — Ciência da Computação, 8º semestre (Aula CP 05, Prática 1)
# Integrantes: Gustavo Landim Dourado
# Ferramentas: Python + biblioteca Lark (lexer="basic") + ipywidgets

# ============================================================
# CÉLULA 0 — PREPARAÇÃO DO AMBIENTE E IMPORTS
# ============================================================
%pip install -q lark ipywidgets

import re
import html
import difflib
from collections import Counter
from datetime import datetime

import lark
from lark import Lark
from lark.exceptions import UnexpectedCharacters

import ipywidgets as widgets
from IPython.display import display, HTML

print(f"✅ Lark versão {lark.__version__} pronto!")
print(f"✅ ipywidgets versão {widgets.__version__} pronto!")

# ============================================================
# CÉLULA 1 — FERRAMENTAS DE VISUALIZAÇÃO
# ============================================================
CORES = {}
for _t in ["RASTREIO", "STATUS", "CEP", "EM", "PESO", "FRETE", "VOLUMES",
           "DESTINATARIO", "TRANSPORTADORA", "TRAJETO"]:
    CORES[_t] = "#1565c0"                                  # palavras reservadas: azul
for _t in ["SITUACAO", "NOME_TRANSP"]:
    CORES[_t] = "#6a1b9a"                                  # grupos de reservadas: roxo
CORES.update({
    "COD_RASTREIO": "#c62828", "CEP_VALOR": "#00838f",
    "DATA": "#4e342e", "HORA": "#4e342e",
    "MEDIDA": "#ef6c00", "NUMERO": "#ef6c00",
    "VALOR": "#2e7d32", "TEXTO": "#558b2f",
    "DOC_CPF": "#ad1457", "EMAIL": "#ad1457",
})

def cor_do_token(tipo):
    return CORES.get(tipo, "#455a64")

def tabela_tokens_html(tokens, exibir=None, titulo="Tabela de tokens"):
    exibir = exibir or [str(t.value) for t in tokens]
    td = "style='padding:2px 10px;border-bottom:1px solid #ddd'"
    linhas = "".join(
        f"<tr><td {td}>{i}</td>"
        f"<td {td}><b style='color:{cor_do_token(t.type)}'>{t.type}</b></td>"
        f"<td {td}><code>{html.escape(v)}</code></td>"
        f"<td {td}>{t.line}</td><td {td}>{t.column}</td></tr>"
        for i, (t, v) in enumerate(zip(tokens, exibir), start=1))
    return (f"<h4 style='margin:4px 0'>{titulo} ({len(tokens)} tokens)</h4>"
            "<table style='border-collapse:collapse;font-family:monospace;font-size:13px'>"
            "<tr style='background:#263238;color:white'>"
            "<th style='padding:4px 10px'>#</th><th style='padding:4px 10px'>TOKEN</th>"
            "<th style='padding:4px 10px'>LEXEMA</th><th style='padding:4px 10px'>LINHA</th>"
            f"<th style='padding:4px 10px'>COLUNA</th></tr>{linhas}</table>")

def texto_colorido_html(texto, tokens, exibir=None):
    exibir = exibir or [texto[t.start_pos:t.end_pos] for t in tokens]
    saida, cursor = "", 0
    for t, v in zip(tokens, exibir):
        saida += html.escape(texto[cursor:t.start_pos])      # espaços e comentários
        cor = cor_do_token(t.type)
        saida += (f"<span title='{t.type}' style='color:{cor};font-weight:bold;"
                  f"border-bottom:2px solid {cor}'>{html.escape(v)}</span>")
        cursor = t.end_pos
    saida += html.escape(texto[cursor:])
    return ("<pre style='background:#fafafa;color:#212121;border:1px solid #ddd;"
            f"padding:10px;font-size:14px;line-height:1.7'>{saida}</pre>"
            "<small>💡 Passe o mouse sobre um lexema para ver o nome do token.</small>")

def erro_lexico_html(texto, erro):
    linhas = texto.splitlines() or [""]
    linha_txt = linhas[erro.line - 1] if erro.line - 1 < len(linhas) else ""
    seta = " " * (erro.column - 1) + "^"
    return ("<div style='background:#ffebee;color:#212121;border-left:5px solid #c62828;padding:10px'>"
            f"<b>❌ ERRO LÉXICO</b> na linha <b>{erro.line}</b>, coluna <b>{erro.column}</b>: "
            f"caractere inesperado <code>{html.escape(repr(erro.char))}</code>"
            f"<pre style='margin:6px 0'>{html.escape(linha_txt)}\n{seta}</pre>"
            f"<small>💡 {html.escape(getattr(erro, 'dica', ''))}</small></div>")

def estatisticas_html(tokens):
    contagem = Counter(t.type for t in tokens)
    barras = "".join(
        f"<div style='margin:2px 0'><code style='display:inline-block;width:150px'>{tipo}</code>"
        f"<span style='display:inline-block;background:{cor_do_token(tipo)};"
        f"height:14px;width:{n * 22}px'></span> {n}</div>"
        for tipo, n in contagem.most_common())
    return f"<h4>Frequência de cada categoria de token</h4>{barras}"

print("✅ Ferramentas de visualização carregadas!")

# ============================================================
# CÉLULA 2 — A GRAMÁTICA DO LEXER (RastreioLang)
# ============================================================
gramatica_rastreio = r"""
start: _token*
_token: RASTREIO | STATUS | CEP | EM | PESO | FRETE | VOLUMES | DESTINATARIO
      | TRANSPORTADORA | TRAJETO | SITUACAO | NOME_TRANSP
      | COD_RASTREIO | CEP_VALOR | DATA | HORA | MEDIDA | VALOR
      | DOC_CPF | EMAIL | NUMERO | TEXTO | SETA

// ---------- 1) Palavras reservadas (prioridade 3) ----------
RASTREIO.3:      /rastreio\b/i
STATUS.3:        /status\b/i
CEP.3:           /cep\b/i
EM.3:            /em\b/i
PESO.3:          /peso\b/i
FRETE.3:         /frete\b/i
VOLUMES.3:       /volumes\b/i
DESTINATARIO.3:  /destinat[aá]rio\b/i
TRANSPORTADORA.3: /transportadora\b/i
TRAJETO.3:       /trajeto\b/i

// ---------- 2) Grupos de reservadas ----------
SITUACAO.4:      /(postado|em[ _]tr[aâ]nsito|saiu[ _]para[ _]entrega|entregue|devolvido|aguardando[ _]retirada)\b/i
NOME_TRANSP.3:   /(correios|sedex|pac|jadlog|loggi|mercado_envios)\b/i

// ---------- 3) Literais estruturados (prioridade 2) ----------
COD_RASTREIO.2:  /[A-Z]{2}\d{9}[A-Z]{2}\b/i        // BR123456789BR
CEP_VALOR.2:     /\d{5}-\d{3}\b/                   // 01310-100
DATA.2:          /\d{2}\/\d{2}\/\d{4}\b/           // 10/09/2026
HORA.2:          /([01]\d|2[0-3]):[0-5]\d\b/        // 08:15 (25:99 não é hora!)
MEDIDA.2:        /\d+(,\d+)?[ ]?(kg|g)\b/i         // 2,5kg  800g  12 kg
VALOR.2:         /R\$ ?(\d{1,3}(\.\d{3})+|\d+),\d{2}\b/   // R$ 24,90  R$ 1.250,00
DOC_CPF.2:       /\d{3}\.\d{3}\.\d{3}-\d{2}\b/      // 123.456.789-09

EMAIL.4:         /[a-z0-9._+-]+@[a-z0-9-]+(\.[a-z0-9-]+)+/i

NUMERO.1:        /\d+/

// ---------- 4) Texto e literal por STRING ----------
TEXTO:           /"[^"\n]*"/                       // "saiu para entrega"
SETA:            "->"                              // literal por string (não é regex)

// ---------- 5) Ruído (descartado) ----------
COMENTARIO: /#[^\n]*/
%ignore COMENTARIO
%ignore /[ \t\r\n]+/
"""

lexer_rastreio = Lark(gramatica_rastreio, parser="lalr", lexer="basic")
_regra = re.search(r"_token:(.*?)\n\n", gramatica_rastreio, re.S).group(1)
TIPOS_DE_TOKEN = re.findall(r"[A-Z_]+", _regra)
print(f"✅ Gramática compilada: {len(TIPOS_DE_TOKEN)} tipos de token (mínimo exigido: 12)")

# ============================================================
# CÉLULA 3 — TOKENIZAR + ERROS COM DICAS DO DOMÍNIO
# ============================================================
PALAVRAS = ["RASTREIO", "STATUS", "CEP", "EM", "PESO", "FRETE", "VOLUMES",
            "DESTINATARIO", "TRANSPORTADORA", "TRAJETO", "POSTADO", "ENTREGUE",
            "DEVOLVIDO", "CORREIOS", "SEDEX", "PAC", "JADLOG", "LOGGI"]

def dica_do_dominio(texto, erro):
    c = erro.char
    resto = texto[erro.pos_in_stream:]

    if c == '"':
        return ('Aspas abertas e não fechadas. Todo texto (status, nome, cidade) precisa '
                'de aspas no começo e no fim, na mesma linha.')
    if resto.startswith("R$"):
        return ('Frete inválido. Use R$ 24,90: vírgula e exatamente 2 dígitos nos centavos '
                '(milhar com ponto: R$ 1.250,00).')
    if c == "$":
        return 'Cifrão sem o "R" na frente. Escreva o frete como R$ 24,90.'
    if c == ",":
        antes = texto[:erro.pos_in_stream]
        if re.search(r"frete\s+\d+$", antes, re.I):
            return 'Frete sem o "R$". Escreva o valor como R$ 24,90.'
        if re.search(r"peso\s+\d+$", antes, re.I):
            return ('Peso inválido. Use número + unidade kg ou g, com no máximo um espaço: '
                    '2,5kg, 2,5 kg ou 800g.')
        return 'Vírgula fora de lugar. Ela só aparece nos decimais de PESO (2,5kg) e de FRETE (R$ 24,90).'
    if c == ":":
        return 'Horário inválido. Use hh:mm entre 00:00 e 23:59 (ex.: 08:15).'
    if c == "/":
        return 'Data inválida. Use dd/mm/aaaa, com 2 dígitos no dia e no mês (ex.: 10/09/2026).'
    if c == "-":
        return ('Hífen fora de lugar. CEP = 5 dígitos + hífen + 3 dígitos (01310-100); '
                'CPF = 123.456.789-09; e o separador de trajeto é ->.')
    if c == ">":
        return 'A seta do trajeto é feita de dois caracteres colados: ->'
    if c == ".":
        return 'Ponto fora de lugar. Decimais usam vírgula (2,5kg); o CPF precisa de todos os pontos e do traço.'
    if c == "@":
        return 'E-mail incompleto. Formato esperado: nome@dominio.com.br'
    if c.isalnum():
        palavra = re.match(r"[\w.+-]+", resto).group(0)
        if resto[len(palavra):].startswith("@"):
            return f'"{palavra}@..." parece um e-mail incompleto. Formato esperado: nome@dominio.com.br'
        if re.fullmatch(r"[A-Za-z]{2}\d+[A-Za-z]*", palavra):
            return (f'"{palavra}" não é um código de rastreio válido. '
                    'Formato: 2 letras + 9 dígitos + 2 letras (ex.: BR123456789BR).')
        sugestao = difflib.get_close_matches(palavra.upper(), PALAVRAS, n=1, cutoff=0.6)
        if sugestao:
            return f'Palavra "{palavra}" não pertence à linguagem. Você quis dizer {sugestao[0]}?'
        return (f'Palavra "{palavra}" não pertence à linguagem de rastreio. '
                'Palavras reservadas: RASTREIO, STATUS, CEP, EM, PESO, FRETE, VOLUMES, '
                'DESTINATARIO, TRANSPORTADORA, TRAJETO.')
    return "Caractere fora do alfabeto da linguagem de rastreio."

def tokenizar_rastreio(texto):
    try:
        return list(lexer_rastreio.lex(texto))
    except UnexpectedCharacters as erro:
        erro.dica = dica_do_dominio(texto, erro)
        raise

# ---- Teste rápido no console ----
entrada = """
RASTREIO BR123456789BR
STATUS "saiu para entrega"
CEP 01310-100
EM 10/09/2026 08:15
"""
print(f"Entrada: {entrada}\n")
for t in tokenizar_rastreio(entrada):
    print(f"  L{t.line} C{t.column:<3} {t.type:<13} -> {t.value!r}")

# ============================================================
# CÉLULA 4 — DO LEXEMA AO VALOR + LGPD + PÓS-PROCESSAMENTO
# ============================================================
def valor_do_frete(lexema):
    return float(lexema.replace("R$", "").strip().replace(".", "").replace(",", "."))

def peso_em_kg(lexema):
    m = re.fullmatch(r"(\d+(?:,\d+)?)\s?(kg|g)", lexema, flags=re.I)
    numero = float(m.group(1).replace(",", "."))
    return numero if m.group(2).lower() == "kg" else numero / 1000

def data_do_lexema(lexema):
    return datetime.strptime(lexema, "%d/%m/%Y").date()

def brl(v):
    return "R$ " + f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def mascarar_cpf(v):   return "***" + v[3:12] + "**"
def mascarar_email(v):
    usuario, dominio = v.split("@")
    return usuario[0] + "***@" + dominio
def mascarar_cep(v):   return v[:5] + "-***"
def mascarar_nome(nome):
    partes = nome.split()
    return " ".join([partes[0]] + [p[0] + "." for p in partes[1:]])

def exibicao_lgpd(tokens, lgpd=True):
    saida, anterior = [], None
    for t in tokens:
        v = str(t.value)
        if lgpd:
            if t.type == "DOC_CPF":
                v = mascarar_cpf(v)
            elif t.type == "EMAIL":
                v = mascarar_email(v)
            elif t.type == "CEP_VALOR":
                v = mascarar_cep(v)
            elif t.type == "TEXTO" and anterior == "DESTINATARIO":
                v = '"' + mascarar_nome(v.strip('"')) + '"'
        saida.append(v)
        anterior = t.type
    return saida

ESPERA = {
    "RASTREIO": ("COD_RASTREIO",), "STATUS": ("SITUACAO", "TEXTO"),
    "CEP": ("CEP_VALOR",), "EM": ("DATA",), "PESO": ("MEDIDA",),
    "FRETE": ("VALOR",), "VOLUMES": ("NUMERO",),
    "TRANSPORTADORA": ("NOME_TRANSP",), "DESTINATARIO": ("TEXTO",),
}
CAMPO = {
    "RASTREIO": "codigo", "STATUS": "status", "CEP": "cep", "EM": "data",
    "PESO": "peso", "FRETE": "frete", "VOLUMES": "volumes",
    "TRANSPORTADORA": "transportadora", "DESTINATARIO": "destinatario",
}

def validar_encomenda(e):
    cod = e.get("codigo")
    if cod and cod[-2:].upper() != "BR":
        e["alertas"].append(f"Código com sufixo {cod[-2:].upper()}: origem fora do Brasil.")
    if "data" in e:
        try:
            data_do_lexema(e["data"])
        except ValueError:
            e["alertas"].append(f"A data {e['data']} não existe no calendário.")
    if "peso" in e and peso_em_kg(e["peso"]) > 30:
        e["alertas"].append(f"Peso de {peso_em_kg(e['peso']):.1f} kg acima de 30 kg.")
    if e.get("status") == "entregue" and "data" not in e:
        e["alertas"].append("Encomenda ENTREGUE sem data de entrega (falta EM dd/mm/aaaa).")

def montar_encomendas(tokens):
    encomendas, orfaos, atual = [], 0, None
    for i, t in enumerate(tokens):
        prox = tokens[i + 1] if i + 1 < len(tokens) else None
        if t.type == "RASTREIO":
            atual = {"linha": t.line, "alertas": []}
            encomendas.append(atual)
        if atual is None:
            orfaos += 1
            continue
        if t.type == "TRAJETO":
            cidades, j = [], i + 1
            while j < len(tokens) and tokens[j].type == "TEXTO":
                cidades.append(tokens[j].value.strip('"'))
                if j + 1 < len(tokens) and tokens[j + 1].type == "SETA":
                    j += 2
                else:
                    break
            if cidades:
                atual["trajeto"] = " → ".join(cidades)
            else:
                atual["alertas"].append(f"Linha {t.line}: depois de TRAJETO eu esperava TEXTO.")
        elif t.type in ESPERA:
            if prox is None or prox.type not in ESPERA[t.type]:
                achou = prox.type if prox else "o fim da entrada"
                atual["alertas"].append(
                    f"Linha {t.line}: depois de {t.type} eu esperava {' ou '.join(ESPERA[t.type])}, "
                    f"mas veio {achou}.")
                continue
            valor = prox.value.strip('"')
            if t.type == "STATUS":
                valor = valor.replace("_", " ").lower()
            atual[CAMPO[t.type]] = valor
            if t.type == "EM" and i + 2 < len(tokens) and tokens[i + 2].type == "HORA":
                atual["hora"] = tokens[i + 2].value
        elif t.type == "DOC_CPF":
            atual["cpf"] = t.value
        elif t.type == "EMAIL":
            atual["email"] = t.value
    for e in encomendas:
        validar_encomenda(e)
    return encomendas, orfaos

def _campo(rotulo, valor):
    return f"{rotulo}: <b>{html.escape(str(valor))}</b><br>" if valor else ""

def encomendas_html(tokens, lgpd=True):
    encomendas, orfaos = montar_encomendas(tokens)
    if not encomendas:
        return "<p>Nenhuma encomenda encontrada (falta a palavra <code>RASTREIO</code>).</p>"
    cartoes = ""
    for n, e in enumerate(encomendas, start=1):
        cep = mascarar_cep(e["cep"]) if lgpd and "cep" in e else e.get("cep")
        nome = mascarar_nome(e["destinatario"]) if lgpd and "destinatario" in e else e.get("destinatario")
        cpf = mascarar_cpf(e["cpf"]) if lgpd and "cpf" in e else e.get("cpf")
        email = mascarar_email(e["email"]) if lgpd and "email" in e else e.get("email")
        peso = f"{peso_em_kg(e['peso']):.3f} kg".replace(".", ",") if "peso" in e else None
        frete = brl(valor_do_frete(e["frete"])) if "frete" in e else None
        quando = " ".join(x for x in [e.get("data"), e.get("hora")] if x) or None
        avisos = "".join(f"<p style='color:#c62828;margin:4px 0'>⚠️ {html.escape(a)}</p>" for a in e["alertas"])
        cartoes += (
            "<div style='font-family:monospace;color:#212121;background:#fffde7;max-width:560px;"
            "border:1px dashed #999;padding:12px;margin:8px 0'>"
            f"<b>📦 Encomenda {n} — {html.escape(e.get('codigo', '(sem código)'))}</b> "
            f"<small>(linha {e['linha']})</small><hr>"
            + _campo("Status", e.get("status")) + _campo("Transportadora", e.get("transportadora"))
            + _campo("Volumes", e.get("volumes")) + _campo("Peso", peso) + _campo("Frete", frete)
            + _campo("Trajeto", e.get("trajeto")) + _campo("CEP de destino", cep)
            + _campo("Data/hora do evento", quando) + _campo("Destinatário", nome)
            + _campo("CPF", cpf) + _campo("E-mail", email) + avisos + "</div>")
    if orfaos:
        cartoes += f"<p>⚠️ {orfaos} token(s) antes do primeiro RASTREIO ficaram de fora.</p>"
    return cartoes

print("✅ Conversões, LGPD e pós-processamento carregados!")

# ============================================================
# CÉLULA 5 — CASOS DE TESTE E VALIDAÇÃO
# ============================================================
CASOS = {
    "✅ Válido 1 — encomenda simples (enunciado)": (
        """RASTREIO BR123456789BR
        STATUS "saiu para entrega"
        CEP 01310-100
        EM 10/09/2026 08:15""",
        "OK"
    ),

    "✅ Válido 2 — completa, multilinha, com comentário e trajeto": (
        """# Pedido #8841 — Mercado Livre
        RASTREIO BR123456789BR
        STATUS em_transito
        TRANSPORTADORA correios VOLUMES 2
        PESO 2,5kg FRETE R$ 24,90
        TRAJETO "São Paulo" -> "Curitiba" -> "Florianópolis"
        DESTINATARIO "Maria Souza" 123.456.789-09 maria.souza@gmail.com
        CEP 01310-100 EM 10/09/2026 08:15""",
        "OK"
    ),

    "✅ Válido 3 — lote de 2 encomendas (minúsculas, milhar, internacional)": (
        """RASTREIO AB987654321BR
        STATUS entregue
        EM 08/09/2026 14:20
        PESO 800g FRETE R$ 18,50

        rastreio cd123456789us
        status postado
        em 09/09/2026 09:00
        peso 12 kg frete R$ 1.250,00""",
        "OK"
    ),

    "✅ Válido 4 — conflito: e-mail que começa com 'em'": (
        'RASTREIO BR123456789BR DESTINATARIO "Ana Lima" em@loja.com.br', "OK"
    ),

    "⚠️ Léxico OK, regra recusa 1 — CEP sem hífen": (
        "RASTREIO BR123456789BR CEP 01310100 STATUS entregue", "OK"
    ),

    "⚠️ Léxico OK, regra recusa 2 — data inexistente e peso alto": (
        "RASTREIO BR123456789BR STATUS postado EM 31/02/2026 10:00 PESO 45kg", "OK"
    ),

    "❌ Inválido 1 — aspas não fechadas": (
        'RASTREIO BR123456789BR\nSTATUS "saiu para entrega', (2, 8, '"')
    ),

    "❌ Inválido 2 — código de rastreio incompleto": (
        "RASTREIO BR12345678BR", (1, 10, "B")
    ),

    "❌ Inválido 3 — peso com unidade errada": (
        "RASTREIO BR123456789BR PESO 2,5 kgs", (1, 30, ",")
    ),

    "❌ Inválido 4 — frete com ponto nos centavos": (
        "RASTREIO BR123456789BR FRETE R$ 24.90", (1, 30, "R")
    ),

    "❌ Inválido 5 — palavra reservada com erro de digitação": (
        "RASTREO BR123456789BR", (1, 1, "R")
    ),

    "❌ Inválido 6 — hora inexistente (25:99)": (
        "RASTREIO BR123456789BR EM 10/09/2026 25:99", (1, 40, ":")
    ),

    "❌ Inválido 7 — e-mail incompleto": (
        'DESTINATARIO "Ana" maria@', (1, 20, "m")
    ),
}

casos_rastreio  = {nome: texto for nome, (texto, _) in CASOS.items()}
esperado_rastreio = {nome: esp for nome, (_, esp) in CASOS.items()}

def testar(tokenizar, casos, esperado):
    acertos = 0
    for nome, texto in casos.items():
        try:
            toks = tokenizar(texto)
            real = "OK"
            resumo = f"{len(toks):>3} tokens: {[t.type for t in toks][:5]}..."
        except UnexpectedCharacters as e:
            real = (e.line, e.column, e.char)
            resumo = f"ERRO na linha {e.line}, coluna {e.column} ({e.char!r})"
        passou = real == esperado[nome]
        acertos += passou
        print(f"{nome[:60]:<60} -> {resumo}  [{'passou' if passou else 'FALHOU'}]")
    print(f"\n{acertos}/{len(casos)} casos com o resultado esperado")
    assert acertos == len(casos), "algum caso divergiu do esperado!"

testar(tokenizar_rastreio, casos_rastreio, esperado_rastreio)

# ============================================================
# CÉLULA 6 — LABORATÓRIO DE PRIORIDADE
# ============================================================
def lexer_com_prioridade(nome, nova):
    g = re.sub(rf"^{nome}\.\d+:", f"{nome}.{nova}:", gramatica_rastreio, flags=re.M)
    return Lark(g, parser="lalr", lexer="basic")

def experimento(nome, nova, texto, lexer=None):
    lexer = lexer or lexer_com_prioridade(nome, nova)
    try:
        toks = " ".join(f"{t.type}({t.value})" for t in lexer.lex(texto))
        resultado = "OK   → " + toks
    except UnexpectedCharacters as e:
        resultado = f"ERRO → linha {e.line}, coluna {e.column}, caractere {e.char!r}"
    print(f"   {nome}.{nova:<2} {resultado}")

print('\nExperimento 1 — EMAIL x reservada EM   | entrada: DESTINATARIO "Ana Lima" em@loja.com.br')
for p in (4, 3, 2):
    experimento("EMAIL", p, 'DESTINATARIO "Ana Lima" em@loja.com.br')

print('\nExperimento 2 — SITUACAO x reservada EM | entrada: STATUS em transito')
for p in (4, 3, 2):
    experimento("SITUACAO", p, "STATUS em transito")

print("\nExperimento 3 — NUMERO x literais numéricos")
for entrada in ["CEP 01310-100", "EM 10/09/2026", "PESO 2,5kg"]:
    print(f"   entrada: {entrada}")
    for p in (1, 2, 3):
        experimento("NUMERO", p, entrada)

print("\nExperimento 4 — o que o \\b faz | entrada: EMPRESA")
sem_b = Lark(gramatica_rastreio.replace(r"EM.3:            /em\b/i", r"EM.3:            /em/i"),
             parser="lalr", lexer="basic")
experimento("EM (sem \\b)", "3", "EMPRESA", lexer=sem_b)
experimento("EM (com \\b)", "3", "EMPRESA", lexer=lexer_rastreio)

# ============================================================
# CÉLULA 7 — INTERFACE (ipywidgets)
# ============================================================
def montar_interface(titulo, tokenizar, casos):
    seletor = widgets.Dropdown(options=list(casos), description="Casos:",
                               layout=widgets.Layout(width="90%"))
    entrada = widgets.Textarea(value=list(casos.values())[0],
                               layout=widgets.Layout(width="95%", height="190px"))
    lgpd = widgets.Checkbox(value=True, description="🔒 Mascarar dados pessoais (LGPD)")
    botao = widgets.Button(description="🔍 Analisar", button_style="primary")
    status = widgets.HTML()
    abas = widgets.Tab(children=[widgets.Output() for _ in range(4)])
    for i, nome in enumerate(["🎨 Colorido", "📋 Tokens", "📦 Encomendas", "📊 Estatística"]):
        abas.set_title(i, nome)

    def analisar(_=None):
        texto = entrada.value
        for aba in abas.children:
            aba.clear_output()
        try:
            tokens = tokenizar(texto)
        except UnexpectedCharacters as erro:
            status.value = erro_lexico_html(texto, erro)
            return
        status.value = (f"<b style='color:#2e7d32'>✅ Análise léxica OK — "
                        f"{len(tokens)} tokens reconhecidos.</b>")
        exibir = exibicao_lgpd(tokens, lgpd.value)
        conteudos = [texto_colorido_html(texto, tokens, exibir),
                     tabela_tokens_html(tokens, exibir),
                     encomendas_html(tokens, lgpd.value),
                     estatisticas_html(tokens)]
        for aba, conteudo in zip(abas.children, conteudos):
            with aba:
                display(HTML(conteudo))

    def escolher(mudanca):
        entrada.value = casos[mudanca["new"]]
        analisar()

    seletor.observe(escolher, names="value")
    lgpd.observe(lambda m: analisar(), names="value")
    botao.on_click(analisar)
    display(widgets.HTML(f"<h3>{titulo}</h3>"), seletor, entrada,
            widgets.HBox([botao, lgpd]), status, abas)
    analisar()

montar_interface("📦 RastreioLang — Analisador Léxico de Encomendas",
                 tokenizar_rastreio, casos_rastreio)